
### Pipeline
-----------
Pipeline completo de previsão de churn de clientes de telecom.

Executa, em sequência:
  1. Data Understanding (EDA)      -> gera gráficos em ./outputs
  2. Data Preparation              -> limpeza e tratamento dos dados
  3. Modelagem - Regressão Logística
  4. Modelagem - Random Forest
  5. Tuning (GridSearchCV)
  6. Comparação final dos modelos  -> resultados salvos em results.json

Uso:
    python src/Pipeline.ipynb

Pré-requisito: rodar a partir da raiz do projeto (onde ficam as pastas
`data/` e `outputs/`), com o arquivo `data/Telecom_Churn.csv` presente.

In [14]:

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme()
sns.set_palette("Accent")
sns.set_style("darkgrid")

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score, balanced_accuracy_score,
                              precision_score, recall_score, f1_score, classification_report,
                              confusion_matrix)

import json, os

RESULTS = {}
IMG = '../images'
os.makedirs(IMG, exist_ok=True)


In [15]:
# ===================== ETAPA 01 - DATA UNDERSTANDING =====================

df = pd.read_csv('../data/Telecom_Churn.csv')


RESULTS['shape'] = df.shape
RESULTS['columns'] = list(df.columns)
RESULTS['dtypes'] = {c: str(t) for c, t in df.dtypes.items()}
RESULTS['nulls'] = df.isna().sum().to_dict()
RESULTS['churn_counts'] = df['Churn'].value_counts().to_dict()
RESULTS['churn_pct'] = (df['Churn'].value_counts(normalize=True) * 100).round(2).to_dict()

# describe numeric
RESULTS['describe'] = df.describe().round(2).to_dict()

# Gráfico 1: distribuição churn
fig, ax = plt.subplots(figsize=(6,4))
churn_counts = df.groupby("Churn")['customerID'].count().reset_index()
sns.barplot(data=churn_counts, x="Churn", y="customerID", ax=ax)
ax.set_title("Distribuição de Clientes por Churn")
ax.set_ylabel("Quantidade de clientes")
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x()+p.get_width()/2, p.get_height()), ha='center', va='bottom')
plt.tight_layout()
plt.savefig(f'{IMG}/01_churn_distribution.png', dpi=110)
plt.close()

# Gráfico 2: churn por tipo de contrato
churn_contract = df.query("Churn == 'Yes'").groupby("Contract")['customerID'].count().reset_index()
fig, ax = plt.subplots(figsize=(7,4))
sns.barplot(data=churn_contract, x="customerID", y="Contract", ax=ax)
ax.set_title("Clientes que Cancelaram (Churn = Yes) por Tipo de Contrato")
ax.set_xlabel("Quantidade de clientes")
plt.tight_layout()
plt.savefig(f'{IMG}/02_churn_by_contract.png', dpi=110)
plt.close()
RESULTS['churn_by_contract'] = churn_contract.set_index('Contract')['customerID'].to_dict()

# Gráfico 3: churn por gênero
churn_gender = df.query("Churn == 'Yes'").groupby("gender")['customerID'].count().reset_index()
fig, ax = plt.subplots(figsize=(6,4))
sns.barplot(data=churn_gender, x="customerID", y="gender", ax=ax)
ax.set_title("Clientes que Cancelaram por Gênero")
ax.set_xlabel("Quantidade de clientes")
plt.tight_layout()
plt.savefig(f'{IMG}/03_churn_by_gender.png', dpi=110)
plt.close()
RESULTS['churn_by_gender'] = churn_gender.set_index('gender')['customerID'].to_dict()

# Gráfico 4: churn por tipo de serviço de internet
churn_internet = df.query("Churn == 'Yes'").groupby("InternetService")['customerID'].count().reset_index()
fig, ax = plt.subplots(figsize=(7,4))
sns.barplot(data=churn_internet, x="customerID", y="InternetService", ax=ax)
ax.set_title("Clientes que Cancelaram por Tipo de Internet")
ax.set_xlabel("Quantidade de clientes")
plt.tight_layout()
plt.savefig(f'{IMG}/04_churn_by_internet.png', dpi=110)
plt.close()
RESULTS['churn_by_internet'] = churn_internet.set_index('InternetService')['customerID'].to_dict()

# Gráfico 5: churn por StreamingTV
churn_streaming = df.query("Churn == 'Yes'").groupby("StreamingTV")['customerID'].count().reset_index()
fig, ax = plt.subplots(figsize=(7,4))
sns.barplot(data=churn_streaming, x="customerID", y="StreamingTV", ax=ax)
ax.set_title("Clientes que Cancelaram por Uso de Streaming TV")
ax.set_xlabel("Quantidade de clientes")
plt.tight_layout()
plt.savefig(f'{IMG}/05_churn_by_streaming.png', dpi=110)
plt.close()
RESULTS['churn_by_streaming'] = churn_streaming.set_index('StreamingTV')['customerID'].to_dict()

# Gráfico 6: pairplot (numéricas) - amostra pra não pesar demais
num_cols = ['tenure', 'MonthlyCharges']
df_pp = df.copy()
df_pp['TotalCharges_num'] = pd.to_numeric(df_pp['TotalCharges'], errors='coerce')
pp = sns.pairplot(df_pp[['tenure', 'MonthlyCharges', 'TotalCharges_num', 'Churn']].dropna(), hue='Churn', diag_kind='hist')
pp.fig.suptitle("Pairplot das variáveis numéricas por Churn", y=1.02)
pp.savefig(f'{IMG}/06_pairplot.png', dpi=110)
plt.close('all')

# Gráfico 7: correlação numérica com histograma de tenure
fig, ax = plt.subplots(figsize=(7,4))
sns.histplot(data=df_pp, x='tenure', hue='Churn', multiple='stack', bins=30, ax=ax)
ax.set_title("Distribuição de Tempo de Contrato (tenure) por Churn")
plt.tight_layout()
plt.savefig(f'{IMG}/07_tenure_hist.png', dpi=110)
plt.close()

print("EDA concluída. Imagens salvas em", IMG)

c:\Users\vivid\anaconda3\Lib\site-packages\seaborn\axisgrid.py:118: UserWarning: The figure layout has changed to tight
  self._figure.tight_layout(*args, **kwargs)


EDA concluída. Imagens salvas em ../images


In [16]:
# ===================== ETAPA 02 - DATA PREPARATION =====================
df_prep = df.copy()

# Corrigir TotalCharges (strings vazias -> NaN -> tratar)
df_prep['TotalCharges'] = df_prep['TotalCharges'].replace(' ', np.nan)
df_prep['TotalCharges'] = pd.to_numeric(df_prep['TotalCharges'])
n_missing_totalcharges = df_prep['TotalCharges'].isna().sum()
RESULTS['missing_totalcharges_found'] = int(n_missing_totalcharges)
# Esses clientes têm tenure = 0 (clientes novos, ainda não cobrados) -> preencher com 0
df_prep['TotalCharges'] = df_prep['TotalCharges'].fillna(0)

# Garantir consistência de rótulo (Churn já vem como Yes/No neste dataset)
df_prep['Churn'] = df_prep['Churn'].replace({0: 'No', 1: 'Yes'})

RESULTS['shape_after_prep'] = df_prep.shape

In [17]:
# ===================== ETAPA 03 - MODELAGEM: REGRESSÃO LOGÍSTICA =====================
X = df_prep.drop(columns=['customerID', 'Churn'])
y_raw = df_prep['Churn']

le = LabelEncoder()
y = le.fit_transform(y_raw)  # No=0, Yes=1
RESULTS['label_mapping'] = {str(k): int(v) for k, v in zip(le.classes_, le.transform(le.classes_))}

X = pd.get_dummies(X)
feature_names = list(X.columns)

mm = MinMaxScaler()
X_scaled = pd.DataFrame(mm.fit_transform(X), columns=feature_names)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.25, random_state=42, stratify=y)
RESULTS['train_shape'] = X_train.shape
RESULTS['test_shape'] = X_test.shape

model_lr = LogisticRegression(max_iter=1000)
lr = model_lr.fit(X_train, y_train)

def eval_model(model, name):
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)
    res = {
        'accuracy_train': round(accuracy_score(y_train, y_pred_train), 4),
        'accuracy_test': round(accuracy_score(y_test, y_pred_test), 4),
        'balanced_accuracy_train': round(balanced_accuracy_score(y_train, y_pred_train), 4),
        'balanced_accuracy_test': round(balanced_accuracy_score(y_test, y_pred_test), 4),
        'precision_test': round(precision_score(y_test, y_pred_test), 4),
        'recall_test': round(recall_score(y_test, y_pred_test), 4),
        'f1_test': round(f1_score(y_test, y_pred_test), 4),
    }
    cm = confusion_matrix(y_test, y_pred_test)
    fig, ax = plt.subplots(figsize=(5,5))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
    disp.plot(ax=ax, values_format='d', cmap='Blues')
    ax.set_title(f"Matriz de Confusão - {name}")
    plt.tight_layout()
    fname = f'{IMG}/cm_{name.lower().replace(" ", "_")}.png'
    plt.savefig(fname, dpi=110)
    plt.close()
    return res

RESULTS['logistic_regression'] = eval_model(lr, "Regressao Logistica")


In [18]:

# ===================== ETAPA 04 - MODELAGEM: RANDOM FOREST =====================
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)
RESULTS['random_forest'] = eval_model(rf, "Random Forest")

# feature importance
importances = pd.Series(rf.feature_importances_, index=feature_names).sort_values(ascending=False).head(15)
fig, ax = plt.subplots(figsize=(8,6))
sns.barplot(x=importances.values, y=importances.index, ax=ax)
ax.set_title("Top 15 Features mais Importantes - Random Forest")
ax.set_xlabel("Importância")
plt.tight_layout()
plt.savefig(f'{IMG}/08_feature_importance.png', dpi=110)
plt.close()

In [19]:

# ===================== ETAPA 05 - TUNING (GridSearchCV) =====================
parameters = {'max_depth': [3, 5, 7, 9, 10],
              'n_estimators': [100, 300, 500]}
grid_search = GridSearchCV(rf, parameters, scoring='accuracy', cv=5, n_jobs=-1)
grid_search.fit(X_train, y_train)

RESULTS['best_params'] = grid_search.best_params_
RESULTS['best_cv_score'] = round(grid_search.best_score_, 4)

rf_tunned = grid_search.best_estimator_
RESULTS['random_forest_tunned'] = eval_model(rf_tunned, "Random Forest Tunned")

# Comparativo final
comparison = pd.DataFrame({
    'Regressão Logística': RESULTS['logistic_regression'],
    'Random Forest': RESULTS['random_forest'],
    'Random Forest (Tuned)': RESULTS['random_forest_tunned'],
}).T
fig, ax = plt.subplots(figsize=(9,5))
comparison[['accuracy_test', 'balanced_accuracy_test', 'f1_test']].plot(kind='bar', ax=ax)
ax.set_title("Comparação de Métricas entre Modelos (Teste)")
ax.set_ylabel("Score")
ax.set_xticklabels(ax.get_xticklabels(), rotation=15)
plt.tight_layout()
plt.savefig(f'{IMG}/09_model_comparison.png', dpi=110)
plt.close()

with open('results.json', 'w') as f:
    json.dump(RESULTS, f, indent=2, ensure_ascii=False, default=str)

print(json.dumps(RESULTS, indent=2, ensure_ascii=False, default=str))


{
  "shape": [
    7043,
    21
  ],
  "columns": [
    "customerID",
    "gender",
    "SeniorCitizen",
    "Partner",
    "Dependents",
    "tenure",
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaperlessBilling",
    "PaymentMethod",
    "MonthlyCharges",
    "TotalCharges",
    "Churn"
  ],
  "dtypes": {
    "customerID": "object",
    "gender": "object",
    "SeniorCitizen": "int64",
    "Partner": "object",
    "Dependents": "object",
    "tenure": "int64",
    "PhoneService": "object",
    "MultipleLines": "object",
    "InternetService": "object",
    "OnlineSecurity": "object",
    "OnlineBackup": "object",
    "DeviceProtection": "object",
    "TechSupport": "object",
    "StreamingTV": "object",
    "StreamingMovies": "object",
    "Contract": "object",
    "PaperlessBilling": "object",
    "PaymentMethod": "ob